In [ ]:
! pip install pypdf
! pip install easyocr
! pip install docling
! pip install google

In [ ]:
import os
from pathlib import Path
from google.colab import drive
from pypdf import PdfReader, PdfWriter

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions

In [ ]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "true"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_DEVICE"] = "cuda"

In [ ]:
drive.mount('/content/drive')
INPUT_DIR = "/content/drive/MyDrive/learningagentbooks"
OUTPUT_DIR = "/content/drive/MyDrive/learningagentmarkdown"

In [ ]:
def shorten_pdf(input_path, output_path, pages):
  reader = PdfReader(input_path)
  writer = PdfWriter()

  for page_num in range(pages):
    writer.add_page(reader.pages[page_num])

  with open(output_path, "wb") as f:
    writer.write(f)

def check_file_names():
    os.makedirs("/content/drive/MyDrive/learningagentbooks", exist_ok=True)
    os.makedirs("/content/drive/MyDrive/learningagentmarkdown", exist_ok=True)

    visible_files = os.listdir("/content/drive/MyDrive/learningagentbooks")
    visible_files = visible_files + os.listdir("/content/drive/MyDrive/learningagentmarkdown")
    if len(visible_files) == 0:
        print("The folder is EMPTY!")
    else:
        for file in visible_files:
            print(f"Found: {file}")

In [ ]:
def convert_with_docling(file_name: str) -> str:
    """Convert a PDF to Markdown using Docling."""
    input_path = os.path.join(INPUT_DIR, file_name)
    output_path = os.path.join(OUTPUT_DIR, Path(file_name).stem + ".md")

    pipeline_options = PdfPipelineOptions(
        do_ocr=True,
        do_table_structure=True,
        do_formula_enrichment=True,
        generate_picture_images=False,
    )
    pipeline_options.ocr_options = EasyOcrOptions(force_full_page_ocr=True)

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    result = converter.convert(input_path)
    document = result.document

    markdown_text = document.export_to_markdown(
        page_break_placeholder="\n\n---\n\n"
    )

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_text)

    return output_path

In [ ]:
input_file_name = "test_snippet.pdf"
if __name__ == "__main__":
    os.makedirs(INPUT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    convert_with_docling(input_file_name)